## Generating Image Captions using a visual LLM model - Utilizando granite-vision-4.1-4b de IBM

In [ ]:
!pip uninstall -y transformers tokenizers peft accelerate
!pip install -U "transformers>=5.8.0" "tokenizers>=0.22.2" "peft>=0.19.1" "accelerate" "pillow>=12.2.0" "safetensors"

import os
os.kill(os.getpid(), 9)

Found existing installation: transformers 5.9.0
Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached transformers-5.9.0-py3-none-any.whl.metadata (33 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
Using cached transformers-5.9.0-py3-none-any.whl (10.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 37.2 MB/s eta 0:00:00


In [1]:
import os
import shutil
import torch
import pandas as pd
from PIL import Image, ImageOps
from transformers import AutoProcessor, AutoModelForImageTextToText

image_dir = "/content/img"
csv_output_path = "/content/image_descriptions.csv"
model_id = "ibm-granite/granite-vision-4.1-4b"

# Remove cached legacy remote-code implementation.
remote_cache_dir = os.path.expanduser(
    "~/.cache/huggingface/modules/transformers_modules/ibm_hyphen_granite"
)
if os.path.exists(remote_cache_dir):
    shutil.rmtree(remote_cache_dir)

CAPTION_PROMPT = """
Generate one concise training caption for this marketing image.

Describe only what is visually visible. Use plain, direct words. Include the image format if clear, such as banner, poster, flyer, social media ad, product photo, website hero image, or package design.

Mention the main subject or product, background, layout, colors, visible text, logos, icons, people, objects, packaging, and composition when relevant. Copy readable text exactly.

Avoid fancy, emotional, subjective, or promotional language. Do not use words like beautiful, premium, elegant, stunning, eye-catching, modern, professional, luxury, clean, or high-quality unless they appear as visible text.

Do not infer the campaign goal, target audience, brand values, mood, intent, product benefits, or hidden meaning. Do not add information that is not visible.

Return only one caption. No bullets. No labels. No explanation. Maximum 50 words.
""".strip()

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = (
    torch.bfloat16
    if device == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float16
    if device == "cuda"
    else torch.float32
)

processor = AutoProcessor.from_pretrained(
    model_id,
    use_fast=False,
)

processor.tokenizer.padding_side = "left"

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype=dtype,
    device_map=device,
).eval()

def load_image(path):
    image = Image.open(path)
    image = ImageOps.exif_transpose(image)
    return image.convert("RGB")

def clean_caption(text):
    text = text.strip().replace("\n", " ")
    text = " ".join(text.split())

    for marker in ["<|assistant|>", "assistant:", "Assistant:"]:
        if marker in text:
            text = text.split(marker)[-1].strip()

    for prefix in [
        "Caption:",
        "caption:",
        "Description:",
        "description:",
        "The image shows",
        "This image shows",
        "The image depicts",
        "This image depicts",
    ]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip(" :.-")

    return text.strip()

def generate_caption(image, prompt, max_new_tokens=100):
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt",
        padding=True,
        do_pad=True,
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    input_len = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0, input_len:]

    caption = processor.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return clean_caption(caption)

valid_exts = (".png", ".jpg", ".jpeg")
image_files = [
    filename
    for filename in sorted(os.listdir(image_dir))
    if filename.lower().endswith(valid_exts)
]

results = []

for filename in image_files:
    image_path = os.path.join(image_dir, filename)

    try:
        image = load_image(image_path)
        caption = generate_caption(
            image=image,
            prompt=CAPTION_PROMPT,
            max_new_tokens=100,
        )

        if not caption:
            caption = "No description generated."

        results.append(
            {
                "file_path": image_path,
                "caption": caption,
            }
        )

        print(f"Processed {filename}: {caption}")

    except Exception as e:
        results.append(
            {
                "file_path": image_path,
                "caption": f"Failed to generate description: {repr(e)}",
            }
        )

        print(f"Error processing {filename}: {repr(e)}")

df_captions = pd.DataFrame(results)
df_captions.to_csv(csv_output_path, index=False)

print(f"Image descriptions saved to {csv_output_path}")
display(df_captions.head())

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


Loading weights:   0%|          | 0/1083 [00:00<?, ?it/s]

Processed 1.autoespar_toyota_corolla_cross_2025.jpg: A banner advertising the Toyota Corolla Cross hybrid electric vehicle, featuring a man walking next to the car in an urban setting.
Processed 10.autoespar_ganadores_cuida_tu_toyota_mama.jpg: A pink and white banner with a man and woman in a car repair shop.
Processed 11.autoespar_toyota_corolla_cross_outlet_2025.jpg: A red Toyota Corolla Cross hybrid electric car is displayed in a promotional image for a 2025 model, with a couple standing beside it. The image is a banner-style advertisement for an outlet store, featuring the text "ÚLTIMAS UNIDADES 2025" and "OUTLET OKM." The background shows a suburban setting with a sunset, and the Toyota logo is visible in the top right corner.
Processed 12.autoespar_toyota_corolla_cross_feliz_dia_madre.jpg: A banner-style advertisement for Toyota's "Día de la Madre" (Mother's Day) campaign, featuring a woman and a child embracing next to a white Toyota Corolla Cross. The image conveys warmth and g

,file_path,caption
0,/content/img/1.autoespar_toyota_corolla_cross_...,A banner advertising the Toyota Corolla Cross ...
1,/content/img/10.autoespar_ganadores_cuida_tu_t...,A pink and white banner with a man and woman i...
2,/content/img/11.autoespar_toyota_corolla_cross...,A red Toyota Corolla Cross hybrid electric car...
3,/content/img/12.autoespar_toyota_corolla_cross...,"A banner-style advertisement for Toyota's ""Día..."
4,/content/img/13.autoespar_toyota_yaris_cross_o...,A poster for a Toyota Yaris Cross with a man s...
